# DATA 715 | MongoDB Atlas: Airbnb Document Explorer

**Live session lab:** Connect to a MongoDB Atlas cluster and explore the real-world document structure of `sample_airbnb.listingsAndReviews` using Python and interactive widgets.

**Learning goals:** inspect BSON documents; query embedded fields and arrays; use projections and aggregations; identify embedded versus referenced data and possible schema anti-patterns.

> This notebook is read-only. It does not insert, modify, or delete documents. A MongoDB Atlas connection and the preloaded sample dataset are required; the notebook itself runs in Google Colab.

## 1. One-time Atlas setup

1. Sign in to [MongoDB Atlas](https://cloud.mongodb.com/) and create or select a cluster.
2. In the cluster options, choose **Load Sample Dataset** if `sample_airbnb` is absent. The collection is named **`listingsAndReviews`**, not `sample_airbnb`.
3. In **Database Access**, create a database user with **read-only access** for this lab (for example, the built-in `readAnyDatabase` role if appropriate, or the `read` role scoped to `sample_airbnb`).
4. In **Network Access**, allow connections from the runtime's public IP. Colab IP addresses can change between sessions. If you temporarily permit all IPs for a class exercise, understand the exposure, use least-privilege credentials, and remove the rule afterward.
5. In **Connect > Drivers > Python**, copy the `mongodb+srv://...` URI. It should include the Atlas cluster host. **Do not paste credentials into this notebook or share a URI containing a password.**

Recommended: use a username and password prompt in the next cell instead of embedding credentials in notebook source. The lab uses TLS and verifies access with `ping`.

[MongoDB sample Airbnb reference](https://www.mongodb.com/docs/manual/sample-data/sample-airbnb/) | [PyMongo connection guide](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/connect/)

## 2. Install packages

In [ ]:
%pip -q install "pymongo[srv]>=4.6,<5" pandas ipywidgets matplotlib

## 3. Connect securely

Paste a **connection-string template** such as `mongodb+srv://cluster0.xxxxx.mongodb.net/?retryWrites=true&w=majority` (no username/password) when prompted. The username and password are entered separately and are not displayed or saved in code cells.

If your URI includes a database name or options, they are preserved. This notebook uses the `sample_airbnb` database explicitly. On connection failure, check the Atlas database user, network access rule, cluster hostname, and password.

In [ ]:
from getpass import getpass
from urllib.parse import urlsplit, urlunsplit, quote
from pymongo import MongoClient
from pymongo.errors import PyMongoError

uri_template = getpass("Atlas MongoDB URI WITHOUT credentials: ").strip()
username = input("Atlas database username: ").strip()
password = getpass("Atlas database password: ")

if not uri_template.startswith(("mongodb+srv://", "mongodb://")):
    raise ValueError("Expected a mongodb+srv:// or mongodb:// connection string")
parts = urlsplit(uri_template)
if "@" in parts.netloc:
    raise ValueError("Use a URI without embedded username/password for this lab")
if not username or not password:
    raise ValueError("Username and password are required")
netloc = f"{quote(username, safe='')}:{quote(password, safe='')}@{parts.netloc}"
uri = urlunsplit((parts.scheme, netloc, parts.path, parts.query, parts.fragment))
client = MongoClient(uri, serverSelectionTimeoutMS=12000, appname="DATA715-Airbnb-Explorer")
try:
    client.admin.command("ping")
    db = client["sample_airbnb"]
    collection = db["listingsAndReviews"]
    print("Connected to MongoDB Atlas successfully.")
    print("Database:", db.name, "| Collection:", collection.name)
    print("Documents:", f"{collection.estimated_document_count():,}")
except PyMongoError as exc:
    client.close()
    raise RuntimeError("Connection failed. Check URI hostname, Atlas network access, user permissions, and password.") from exc
finally:
    # Avoid leaving a cleartext URI in a notebook variable; client retains connection internally.
    del uri, password, netloc

## 4. Inspect a document

MongoDB documents can contain **nested objects**, **arrays**, dates, and Decimal128 fields. The `reviews` array may be very large, so the first preview excludes it.

In [ ]:
from bson import json_util
from IPython.display import display, JSON

sample = collection.find_one({}, {"reviews": 0})
if sample is None:
    raise RuntimeError("No documents found. Check that you loaded the Atlas sample dataset.")
print("Top-level fields:", ", ".join(sorted(sample.keys())))
display(JSON(json_util.loads(json_util.dumps(sample)), expanded=False))

## 5. Browse a compact table

Projection selects the fields returned, reducing the size and complexity of the results. Access nested fields with dot notation.

In [ ]:
import pandas as pd

PREVIEW_PROJECTION = {
    "name": 1, "property_type": 1, "room_type": 1, "bedrooms": 1,
    "beds": 1, "price": 1, "address.market": 1,
    "address.country": 1, "review_scores.review_scores_rating": 1,
    "number_of_reviews": 1
}

def compact_listing(d):
    return {
        "_id": d.get("_id"),
        "name": d.get("name"),
        "market": d.get("address", {}).get("market"),
        "country": d.get("address", {}).get("country"),
        "room_type": d.get("room_type"),
        "bedrooms": d.get("bedrooms"),
        "price": float(d["price"].to_decimal()) if d.get("price") is not None else None,
        "rating": d.get("review_scores", {}).get("review_scores_rating"),
        "review_count": d.get("number_of_reviews")
    }

rows = [compact_listing(doc) for doc in collection.find({}, PREVIEW_PROJECTION).limit(12)]
display(pd.DataFrame(rows))

## 6. Interactive MongoDB explorer

Choose a market, room type, minimum bedrooms, maximum nightly price, and result count, then press **Search**. Select a result to inspect its full document. For readability, the document viewer defaults to the first 3 reviews and reports the total review count.

This explorer uses read-only `find()` queries, projections, and bounded result counts.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

# Pull only distinct values needed for dropdowns.
markets = sorted(x for x in collection.distinct("address.market") if isinstance(x, str) and x)
room_types = sorted(x for x in collection.distinct("room_type") if isinstance(x, str) and x)
market_w = widgets.Dropdown(options=[("All markets", "")] + [(v, v) for v in markets], description="Market:", style={"description_width": "110px"})
room_w = widgets.Dropdown(options=[("All room types", "")] + [(v, v) for v in room_types], description="Room:", style={"description_width": "110px"})
beds_w = widgets.IntSlider(value=0, min=0, max=10, step=1, description="Min bedrooms:", style={"description_width": "110px"})
price_w = widgets.IntSlider(value=500, min=25, max=1500, step=25, description="Max price:", style={"description_width": "110px"})
limit_w = widgets.IntSlider(value=10, min=1, max=30, step=1, description="Results:", style={"description_width": "110px"})
search_button = widgets.Button(description="Search", button_style="primary", icon="search")
results_out = widgets.Output()
selected_w = widgets.Dropdown(options=[], description="Document:", style={"description_width": "110px"}, layout=widgets.Layout(width="95%"))
show_button = widgets.Button(description="View selected document", icon="file-code-o")
detail_out = widgets.Output()
_current_docs = {}

def on_search(_):
    global _current_docs
    q = {}
    if market_w.value: q["address.market"] = market_w.value
    if room_w.value: q["room_type"] = room_w.value
    if beds_w.value: q["bedrooms"] = {"$gte": beds_w.value}
    q["price"] = {"$lte": price_w.value}
    documents = list(collection.find(q, PREVIEW_PROJECTION).limit(limit_w.value))
    _current_docs = {str(doc["_id"]): doc for doc in documents}
    selected_w.options = [(f'{doc.get("name", "(unnamed)")[:65]} | {doc["_id"]}', str(doc["_id"])) for doc in documents]
    with results_out:
        clear_output(wait=True)
        print("Filter:", json_util.dumps(q))
        print("Showing:", len(documents), "documents (up to the selected limit)")
        display(pd.DataFrame([compact_listing(doc) for doc in documents]))
    with detail_out: clear_output(wait=True)

def on_detail(_):
    if not selected_w.value: return
    doc = collection.find_one({"_id": selected_w.value})
    if doc is None: return
    total_reviews = len(doc.get("reviews", []))
    doc["reviews"] = doc.get("reviews", [])[:3]
    with detail_out:
        clear_output(wait=True)
        print("Full listing structure; reviews shown:", min(total_reviews, 3), "of", total_reviews)
        display(JSON(json_util.loads(json_util.dumps(doc)), expanded=False))

search_button.on_click(on_search)
show_button.on_click(on_detail)
display(widgets.VBox([
    widgets.HBox([market_w, room_w]),
    widgets.HBox([beds_w, price_w]),
    widgets.HBox([limit_w, search_button]), results_out,
    widgets.HBox([selected_w, show_button]), detail_out
]))
on_search(None)

## 7. Query nested documents and arrays

This query finds entire homes in a selected market that include Wi-Fi in the `amenities` array. Notice that querying an array for a single matching value does not require an explicit loop.

In [ ]:
query = {
    "address.market": "Porto",
    "room_type": "Entire home/apt",
    "amenities": "Wifi"
}
result = list(collection.find(query, PREVIEW_PROJECTION).limit(10))
display(pd.DataFrame([compact_listing(d) for d in result]))

### Find a listing with embedded reviews

A single document may contain a list of review subdocuments. Projection with `$slice` limits how much of the array is returned.

In [ ]:
review_example = collection.find_one(
    {"reviews.0": {"$exists": True}},
    {"name": 1, "number_of_reviews": 1, "reviews": {"$slice": 2}}
)
if review_example:
    display(JSON(json_util.loads(json_util.dumps(review_example)), expanded=False))

## 8. Aggregation pipeline: number of listings by market

Pipelines combine `$match`, `$group`, `$sort`, and `$limit`. These operations execute in MongoDB rather than transferring the whole collection into Colab.

In [ ]:
pipeline = [
    {"$match": {"address.market": {"$type": "string"}}},
    {"$group": {"_id": "$address.market", "listings": {"$sum": 1}}},
    {"$sort": {"listings": -1}},
    {"$limit": 12}
]
market_counts = pd.DataFrame(list(collection.aggregate(pipeline)))
display(market_counts.rename(columns={"_id": "market"}))
if not market_counts.empty:
    ax = market_counts.sort_values("listings").plot.barh(x="_id", y="listings", legend=False, figsize=(9, 5), title="Airbnb sample listings by market")
    ax.set_xlabel("Number of listings")
    ax.set_ylabel("Market")

## 9. Aggregation pipeline: average price by room type

Prices are stored as MongoDB Decimal128 values. Convert the aggregate result to `float` for display only. Rates are from the sample data and should not be interpreted as current Airbnb prices.

In [ ]:
price_pipeline = [
    {"$match": {"price": {"$type": "decimal"}}},
    {"$group": {"_id": "$room_type", "listings": {"$sum": 1}, "avg_price": {"$avg": "$price"}}},
    {"$sort": {"listings": -1}}
]
price_rows = list(collection.aggregate(price_pipeline))
price_table = pd.DataFrame([{
    "room_type": d["_id"], "listings": d["listings"],
    "avg_price": round(float(d["avg_price"].to_decimal()), 2)
} for d in price_rows])
display(price_table)

## 10. Schema explorer: field frequencies and data types

Document databases do not require every document to have identical fields. This sample inspects up to 200 documents, including nested fields, without scanning the full embedded reviews arrays.

In [ ]:
from collections import Counter, defaultdict

seen = Counter()
field_types = defaultdict(Counter)

def walk_fields(obj, prefix=""):
    if not isinstance(obj, dict): return
    for key, value in obj.items():
        name = f"{prefix}.{key}" if prefix else key
        seen[name] += 1
        field_types[name][type(value).__name__] += 1
        if isinstance(value, dict): walk_fields(value, name)

schema_docs = list(collection.find({}, {"reviews": 0}).limit(200))
for d in schema_docs: walk_fields(d)
schema_table = pd.DataFrame([
    {"field": field, "present_in_sample": count, "pct": round(count / len(schema_docs) * 100, 1),
     "observed_types": ", ".join(f"{t}:{n}" for t, n in field_types[field].items())}
    for field, count in seen.items()
]).sort_values(["pct", "field"], ascending=[False, True]) if schema_docs else pd.DataFrame()
print("Documents inspected:", len(schema_docs))
display(schema_table.head(60))

## 11. Index explorer and query plan

MongoDB indexes can speed up selective queries, but unnecessary indexes consume space and add write overhead. Inspect the indexes defined on this collection and examine the plan for a basic equality query.

In [ ]:
index_rows = []
for idx in collection.list_indexes():
    index_rows.append({"name": idx["name"], "keys": str(list(idx["key"].items()))})
display(pd.DataFrame(index_rows))

plan = collection.find({"name": "Ribeira Charming Duplex"}, {"name": 1}).explain()
print("Winning plan:")
display(JSON(json_util.loads(json_util.dumps(plan.get("queryPlanner", {}).get("winningPlan", {}))), expanded=False))

## 12. Discussion: embedding, referencing, and anti-patterns

Work in pairs or small groups. Refer to a sample document and its `host`, `address`, `amenities`, and `reviews` fields.

1. **Embedding:** Why is `address` naturally embedded within a listing?
2. **One-to-many:** Is embedding every `review` always the best choice? What happens as review counts grow?
3. **Referencing:** If the same `host` owns 50 listings, what data might be duplicated? When would a separate `hosts` collection make sense?
4. **Arrays:** How could an unbounded `reviews` array become a massive-array anti-pattern?
5. **Indexes:** Which two frequently used filters would you consider for a compound index? What query plan evidence would you want?
6. **Schema differences:** Which fields appear in only some sampled documents? Why might that be acceptable?

**Mini-challenge:** Change the filter in Section 7, and add an aggregation to compute average `review_scores.review_scores_rating` by `property_type`. Explain any missing-value handling.

**Deliverable:** Share one example document, a PyMongo query, and one design tradeoff you identified.

## 13. Optional: close the connection

Run this cell when finished. If you close the connection, rerun the connection cell before exploring again.

In [ ]:
client.close()
print("MongoDB connection closed.")

### Notes for instructors

- **Prerequisites:** an Atlas account, a cluster with the sample dataset loaded, a read-only database user, and a temporary network access rule for the Colab runtime.
- **No database writes:** all exercises use `find`, `distinct`, `aggregate`, `list_indexes`, and `explain`.
- **Availability:** the full lab needs an active connection; results are intentionally not fabricated or cached.
- **Credential handling:** prompts hide the password, but use a dedicated, least-privilege training account. Never save or share notebooks containing a credential-bearing URI.
- **Troubleshooting:** If connection times out, check Network Access and whether the cluster is awake. If authentication fails, check the database user and password. If the collection is missing, load the Atlas sample dataset.

Sources: [MongoDB sample Airbnb dataset](https://www.mongodb.com/docs/manual/sample-data/sample-airbnb/), [MongoDB sample data instructions](https://www.mongodb.com/docs/manual/sample-data/), [PyMongo connection documentation](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/connect/).